# 01. Sen1Floods11 Dataset Analysis & Exploration

This notebook provides statistical exploration, backscatter distribution profiling, and class imbalance analysis for the **Sen1Floods11 Sentinel-1 SAR** flood segmentation dataset (including Northeast India / global subsets).

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from fine_tune.datasets.sen1floods_dataset import Sen1FloodsDataset, normalize_sar
from fine_tune.datasets.transforms import get_transforms
from fine_tune.scripts.prepare_dataset import create_synthetic_sen1floods_dataset

## 1. Load Dataset Splits

In [ ]:
data_root = PROJECT_ROOT / "fine_tune" / "data" / "sample_sen1floods11"
if not data_root.exists():
    create_synthetic_sen1floods_dataset(output_dir=data_root)

train_ds = Sen1FloodsDataset(data_root=data_root, split="train", augment=False)
val_ds = Sen1FloodsDataset(data_root=data_root, split="valid", augment=False)
test_ds = Sen1FloodsDataset(data_root=data_root, split="test", augment=False)

print(f"Train Samples:      {len(train_ds)}")
print(f"Validation Samples: {len(val_ds)}")
print(f"Test Samples:       {len(test_ds)}")
print(f"Total Samples:      {len(train_ds) + len(val_ds) + len(test_ds)}")

## 2. SAR Backscatter Statistics & Histograms (VV & VH)

In [ ]:
vv_vals = []
vh_vals = []
flood_pixels = 0
non_flood_pixels = 0

for i in range(len(train_ds)):
    img, msk, _ = train_ds[i]
    img_np = img.numpy()
    msk_np = msk.numpy()
    
    vv_vals.append(img_np[0].flatten())
    vh_vals.append(img_np[1].flatten())
    
    valid = (msk_np != -1) & (msk_np != 255)
    flood_pixels += np.sum((msk_np == 1) & valid)
    non_flood_pixels += np.sum((msk_np == 0) & valid)

vv_all = np.concatenate(vv_vals)
vh_all = np.concatenate(vh_vals)

total_valid = flood_pixels + non_flood_pixels
print(f"Total Valid Pixels:     {total_valid:,}")
print(f"Flood Pixels:           {flood_pixels:,} ({flood_pixels / total_valid * 100:.2f}%)")
print(f"Non-Flood Pixels:       {non_flood_pixels:,} ({non_flood_pixels / total_valid * 100:.2f}%)")
print(f"Class Imbalance Ratio:  1 : {non_flood_pixels / max(1, flood_pixels):.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(vv_all, bins=50, color='royalblue', alpha=0.7, edgecolor='black')
axes[0].set_title("Normalized Sentinel-1 VV Distribution")
axes[0].set_xlabel("Normalized Value [0, 1]")
axes[0].set_ylabel("Pixel Count")
axes[0].grid(True, alpha=0.3)

axes[1].hist(vh_all, bins=50, color='darkorange', alpha=0.7, edgecolor='black')
axes[1].set_title("Normalized Sentinel-1 VH Distribution")
axes[1].set_xlabel("Normalized Value [0, 1]")
axes[1].set_ylabel("Pixel Count")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Visual Sample Inspection (SAR False-Color & Mask Pairs)

In [ ]:
num_samples = min(4, len(train_ds))
fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))

for idx in range(num_samples):
    img, msk, meta = train_ds[idx]
    img_np = img.numpy()
    msk_np = msk.numpy()
    
    vv = img_np[0]
    vh = img_np[1]
    ratio = np.clip(vv / (vh + 1e-4), 0.0, 1.0)
    rgb = np.stack([vv, vh, ratio], axis=-1)
    
    # SAR False Color
    axes[idx, 0].imshow(rgb)
    axes[idx, 0].set_title(f"{meta['sample_id']} - SAR RGB")
    axes[idx, 0].axis("off")
    
    # VV Channel
    axes[idx, 1].imshow(vv, cmap="gray")
    axes[idx, 1].set_title("VV Polarization")
    axes[idx, 1].axis("off")
    
    # Ground Truth Mask
    axes[idx, 2].imshow(msk_np, cmap="Blues", vmin=0, vmax=1)
    axes[idx, 2].set_title("Ground Truth Flood Mask")
    axes[idx, 2].axis("off")

plt.tight_layout()
plt.show()